# Lesson 17 — Spark DataFrames & Spark SQL



## 1. SparkSession

Bu SparkSession obyekti bütün lab boyu istifadə ediləcək giriş nöqtəsidir.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
import pandas as pd
import time

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)



ModuleNotFoundError: No module named 'pandas'

## 2. RDD vs DataFrame vs Dataset 




In [76]:

rdd = spark.sparkContext.parallelize([(1, "A", 245.5), (2, "B", 88.1), (3, "A", 512.75)])
print("RDD type:", type(rdd))
print("RDD :", rdd.collect())

df_from_rdd = rdd.toDF(["id", "category", "amount"])
print("\nDataFrame type:", type(df_from_rdd))
df_from_rdd.printSchema()
df_from_rdd.show()

RDD type: <class 'pyspark.rdd.RDD'>
RDD : [(1, 'A', 245.5), (2, 'B', 88.1), (3, 'A', 512.75)]

DataFrame type: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)

+---+--------+------+
| id|category|amount|
+---+--------+------+
|  1|       A| 245.5|
|  2|       B|  88.1|
|  3|       A|512.75|
+---+--------+------+



## 3. DataFrame 



In [77]:
base_path = "s3a://matrix/lesson20_base_parquet"
df = spark.read.parquet(base_path)
df.printSchema()
df.show(5)
print("count:", df.count())

root
 |-- category: integer (nullable = true)
 |-- amount: double (nullable = true)

+--------+------------------+
|category|            amount|
+--------+------------------+
|       0| 5.096502963909955|
|       0| 603.1354326366278|
|       0| 575.3844817706744|
|       1|102.88761956541637|
|       0| 497.6186788268403|
+--------+------------------+
only showing top 5 rows

count: 1000000


In [51]:
sample_df = spark.createDataFrame(
    [(1, "A", 245.5), (2, "B", 88.1), (3, "A", 512.75), (4, "C", None)],
    ["id", "category", "amount"]
)
sample_df.show()

+---+--------+------+
| id|category|amount|
+---+--------+------+
|  1|       A| 245.5|
|  2|       B|  88.1|
|  3|       A|512.75|
|  4|       C|  NULL|
+---+--------+------+



## 4. Read Modes və Schema

Qəsdən "korrupt" bir CSV yaradıb, üç fərqli read mode-un davranışını müşahidə edək.


In [52]:
# Korrupt CSV
bad_csv_lines = [
    "category,amount",
    "A,245.5",
    "B,88.1",
    "C,not_a_number,extra_column"
]

spark.sparkContext.parallelize(bad_csv_lines, numSlices=1).saveAsTextFile("s3a://matrix/lesson21_bad_data_csv")

schema = StructType([
    StructField("category", StringType()),
    StructField("amount", DoubleType())
])


In [57]:
# PERMISSIVE (default) 
df_permissive = (spark.read.schema(schema)
                  .option("header", True)
                  .option("mode", "PERMISSIVE")
                  .csv("s3a://matrix/lesson21_bad_data_csv"))
print("PERMISSIVE:")
df_permissive.show()

PERMISSIVE:
+--------+------+
|category|amount|
+--------+------+
|       A| 245.5|
|       B|  88.1|
|       C|  NULL|
+--------+------+



In [58]:
# DROPMALFORMED 
df_drop = spark.read.schema(schema).option("header", True).option("mode", "DROPMALFORMED").csv("s3a://matrix/lesson21_bad_data_csv")
print("DROPMALFORMED:")
df_drop.show()

DROPMALFORMED:
+--------+------+
|category|amount|
+--------+------+
|       A| 245.5|
|       B|  88.1|
+--------+------+



In [59]:
# FAILFAST 
try:
    df_fail = spark.read.schema(schema).option("header", True).option("mode", "FAILFAST").csv("s3a://matrix/lesson21_bad_data_csv")
    df_fail.show()
except Exception as e:
    print("error:", type(e).__name__)

error: Py4JJavaError


##  JSON formatından oxumaq



In [69]:
# 1) JSON
json_lines_content = [
    '{"category": "A", "amount": 245.5}',
    '{"category": "B", "amount": 88.1}',
    '{"category": "A", "amount": 512.75}'
]
spark.sparkContext.parallelize(json_lines_content, numSlices=1) \
    .saveAsTextFile("s3a://matrix/lesson21_orders_lines_json")

df_json = spark.read.json("s3a://matrix/lesson21_orders_lines_json")
df_json.printSchema()
df_json.show()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)

+------+--------+
|amount|category|
+------+--------+
| 245.5|       A|
|  88.1|       B|
|512.75|       A|
+------+--------+



In [70]:
# 2) Pretty-printed JSON array — multiLine=True LAZIMDIR, yoxsa sxem səhv oxunar
json_array_content = [
    "[",
    '  {"category": "A", "amount": 245.5},',
    '  {"category": "B", "amount": 88.1}',
    "]"
]
spark.sparkContext.parallelize(json_array_content, numSlices=1) \
    .saveAsTextFile("s3a://matrix/lesson21_orders_array_json")   

# multiLine=False (default) 
df_wrong = spark.read.json("s3a://matrix/lesson21_orders_array_json")
print("multiLine=False count:", df_wrong.count())

# correct
df_correct = spark.read.option("multiLine", True).json("s3a://matrix/lesson21_orders_array_json")
print("multiLine=True count:", df_correct.count())
df_correct.show()

# JSON yazmaq — DataFrame.write.json() Spark-ın öz writer-idir, düzgün işləyir
df_json.write.mode("overwrite").json("s3a://matrix/lesson21_orders_json")

multiLine=False count: 4
multiLine=True count: 2
+------+--------+
|amount|category|
+------+--------+
| 245.5|       A|
|  88.1|       B|
+------+--------+



In [62]:
df_wrong.show()

+---------------+------+--------+
|_corrupt_record|amount|category|
+---------------+------+--------+
|              [|  NULL|    NULL|
|           NULL| 245.5|       A|
|           NULL|  88.1|       B|
|              ]|  NULL|    NULL|
+---------------+------+--------+



## 5. Spark SQL 

Eyni sualı həm DataFrame API, həm də SQL ilə soruşub nəticələri müqayisə edək.


In [71]:
df.createOrReplaceTempView("orders")

df_api = (
    df.filter(F.col("amount") > 100)
      .groupBy("category")
      .agg(F.sum("amount").alias("total"))
      .orderBy(F.desc("total"))
)

df_sql = spark.sql("""
    SELECT category, SUM(amount) AS total
    FROM orders
    WHERE amount > 100
    GROUP BY category
    ORDER BY total DESC
""")

print("DataFrame API :")
df_api.show()
print("Spark SQL :")
df_sql.show()

print("DataFrame API physical plan:")
df_api.explain()
print("\nSpark SQL physical plan:")
df_sql.explain()


DataFrame API :
+--------+-------------------+
|category|              total|
+--------+-------------------+
|       0|9.895796792979302E7|
|       4|9.894529874387953E7|
|       2|9.885599765183817E7|
|       3|9.880517413477829E7|
|       1|9.876368487696871E7|
+--------+-------------------+

Spark SQL :
+--------+-------------------+
|category|              total|
+--------+-------------------+
|       0|9.895796792979302E7|
|       4|9.894529874387953E7|
|       2|9.885599765183817E7|
|       3|9.880517413477829E7|
|       1|9.876368487696871E7|
+--------+-------------------+

DataFrame API physical plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total#1140 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total#1140 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2167]
      +- HashAggregate(keys=[category#732], functions=[sum(amount#733)])
         +- Exchange hashpartitioning(category#732, 200), ENSURE_REQUIREMENTS, [plan_id=2164]
           

## 6. Transformations & Actions (Slayd 8)

`collect()`-i diqqətlə istifadə et — böyük DataFrame üzərində Driver-i çökdürə bilər.


In [ ]:
# Transformation (lazy)
df2 = (
    df.select("category", "amount")
      .withColumn("amount_doubled", F.col("amount") * 2)
      .drop("amount_doubled")
)

# Action-lar
print("count():", df2.count())
print("first():", df2.first())
print("take(3):", df2.take(3))
df2.show(3)

## 7. Filtering (Slayd 9)

`and`/`or` YOX, `&`/`|`/`~` və mötərizə.


In [72]:
result1 = df.filter((F.col("amount") > 100) & (F.col("category") == "A"))
print("Filter 1 (amount>100 AND category=A):", result1.count())

# isin və isNotNull
result2 = df.filter(F.col("category").isin("A", "B"))
print("Filter 2 (category in A,B):", result2.count())

result3 = sample_df.filter(F.col("amount").isNotNull())
print("Filter 3 (amount not null, sample_df üzərində):", result3.count())

Filter 1 (amount>100 AND category=A): 0
Filter 2 (category in A,B): 0
Filter 3 (amount not null, sample_df üzərində): 3


## 8. Aggregations

Bir neçə aqreqat funksiyasını eyni `agg()` çağırışında birləşdirək.


In [ ]:
agg_result = (
    df.groupBy("category")
      .agg(
          F.count("*").alias("cnt"),
          F.sum("amount").alias("total"),
          F.avg("amount").alias("avg_amt"),
          F.min("amount").alias("min_amt"),
          F.max("amount").alias("max_amt")
      )
      .orderBy(F.desc("total"))
)
agg_result.show()

## 9. Built-in Functions (Slayd 11)

Bir neçə kateqoriyadan nümunə: string, date, riyazi, şərti.


In [73]:
demo_df = spark.createDataFrame(
    [("  Alice  ", "2024-01-15", -12.5), ("Bob", "2024-03-22", 45.0)],
    ["name", "signup_date", "balance"]
)

result = (
    demo_df
    .withColumn("name_clean", F.trim(F.col("name")))                     # string
    .withColumn("signup_year", F.year(F.col("signup_date")))             # date
    .withColumn("balance_abs", F.abs(F.col("balance")))                  # riyazi
    .withColumn("status", F.when(F.col("balance") < 0, "debt")           # şərti
                            .otherwise("positive"))
)
result.show(truncate=False)

+---------+-----------+-------+----------+-----------+-----------+--------+
|name     |signup_date|balance|name_clean|signup_year|balance_abs|status  |
+---------+-----------+-------+----------+-----------+-----------+--------+
|  Alice  |2024-01-15 |-12.5  |Alice     |2024       |12.5       |debt    |
|Bob      |2024-03-22 |45.0   |Bob       |2024       |45.0       |positive|
+---------+-----------+-------+----------+-----------+-----------+--------+



## 10. UDF vs Built-in



In [ ]:
from pyspark.sql.types import StringType

# 1) Standart UDF 
@F.udf(returnType=StringType())
def categorize_udf(amount):
    return "high" if amount is not None and amount > 500 else "low"

t0 = time.time()
df.withColumn("tier", categorize_udf(F.col("amount"))).count()
print(f"Standart UDF: {time.time() - t0:.3f} san.")

# 2) Pandas UDF 
@F.pandas_udf("string")
def categorize_pandas(amount: pd.Series) -> pd.Series:
    return amount.apply(lambda x: "high" if x is not None and x > 500 else "low")

t0 = time.time()
df.withColumn("tier", categorize_pandas(F.col("amount"))).count()
print(f"Pandas UDF: {time.time() - t0:.3f} san.")

# 3) Built-in 
t0 = time.time()
df.withColumn("tier", F.when(F.col("amount") > 500, "high").otherwise("low")).count()
print(f"Built-in (when/otherwise): {time.time() - t0:.3f} san.")

Standart UDF: 0.224 san.
Pandas UDF: 0.196 san.
Built-in (when/otherwise): 0.190 san.


## 10b. UDF  Spark SQL




In [ ]:

spark.udf.register("categorize_sql", categorize_udf)

# spark.udf.register("categorize_pandas_sql", categorize_pandas)

df.createOrReplaceTempView("orders_udf_demo")

spark.sql("""
    SELECT
        category,
        amount,
        categorize_sql(amount)        AS tier_standard_udf
     --   categorize_pandas_sql(amount) AS tier_pandas_udf
    FROM orders_udf_demo
    LIMIT 10
""").show()

+--------+------------------+-----------------+
|category|            amount|tier_standard_udf|
+--------+------------------+-----------------+
|       0| 5.096502963909955|              low|
|       0| 603.1354326366278|             high|
|       0| 575.3844817706744|             high|
|       1|102.88761956541637|              low|
|       0| 497.6186788268403|              low|
|       3| 882.9202805075074|             high|
|       2| 811.9803481136765|             high|
|       3|191.95085767330067|              low|
|       4|38.459081662433945|              low|
|       4|509.00763389015424|             high|
+--------+------------------+-----------------+



## 11. Write Modes və partitionBy (Slayd 15)

Əvvəlcə hər 4 write mode-u kiçik, sadə bir path üzərində ayrı-ayrı görək, sonra `category` üzrə partition-lanmış "əsl" nümunəyə keçək.


In [1]:
small_path = "s3a://matrix/lesson21_write_modes_demo"

# 1) overwrite — mövcud data-nı SİLİB yenisini yazır
sample_df.write.mode("overwrite").parquet(small_path)
print("overwrite sonrası sətir sayı:", spark.read.parquet(small_path).count())

sample_df.limit(1).write.mode("overwrite").parquet(small_path)
print("2-ci overwrite (1 sətirlik data ilə) sonrası sətir sayı:", spark.read.parquet(small_path).count())

NameError: name 'sample_df' is not defined

In [ ]:
# 2) append — mövcud data-nın ÜSTÜNƏ əlavə edir
sample_df.write.mode("overwrite").parquet(small_path)   # təmiz başlanğıc
sample_df.write.mode("append").parquet(small_path)


append sonrası sətir sayı (2x olmalıdır): 8


In [74]:
# 3) ignore 
sample_df.write.mode("overwrite").parquet(small_path)  
sample_df.limit(1).write.mode("ignore").parquet(small_path)


In [ ]:
# 4) error / errorifexists (DEFAULT) — path artıq varsa, exception atır
try:
    sample_df.write.mode("error").parquet(small_path)
except Exception as e:
    print("Gözlənilən xəta tutuldu:", type(e).__name__)

try:
    sample_df.write.parquet(small_path)   # mode() çağırılmayıb
except Exception as e:
    print("Default mode = error", type(e).__name__)

Gözlənilən xəta tutuldu: AnalysisException
Default mode = error, eyni xəta: AnalysisException


In [ ]:
partitioned_path = "s3a://matrix/lesson21_orders_partitioned"

(df.write
   .mode("overwrite")
   .partitionBy("category")
   .parquet(partitioned_path))

print("Yazıldı:", partitioned_path)

df_read_back = spark.read.parquet(partitioned_path)
df_read_back.printSchema()
df_read_back.filter(F.col("category") == "A").show(5)

Yazıldı: s3a://matrix/lesson21_orders_partitioned
root
 |-- amount: double (nullable = true)
 |-- category: integer (nullable = true)

+------+--------+
|amount|category|
+------+--------+
+------+--------+



In [ ]:
spark.stop()